# Layered Internal Validation Metrics

This notebook validates the current layered mapping/reference outputs. It replaces the older single-pipeline validation view with Layer 1 nutrient candidates, Layer 2 OpenFoodFacts candidates, the combined HITL queue, and the final canonical-food feature matrix.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs'
LAYERED = OUT / 'layered'
REFERENCE = OUT / 'reference'
VALIDATION = OUT / 'validation'
VALIDATION.mkdir(parents=True, exist_ok=True)

## Load Layered Outputs

In [2]:
layer1 = pd.read_csv(LAYERED / 'layer1_nutrient_candidates.csv')
layer2 = pd.read_csv(LAYERED / 'layer2_openfoodfacts_candidates.csv')
hitl = pd.read_csv(LAYERED / 'hitl_review_queue.csv')
feature = pd.read_csv(REFERENCE / 'canonical_food_feature_matrix.csv')
summary = json.loads((LAYERED / 'layered_pipeline_summary.json').read_text())
summary['reference']

{'outputs': {'canonical_food_master': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_master.csv',
  'canonical_food_nutrient_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_nutrient_reference.csv',
  'canonical_food_processing_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_processing_reference.csv',
  'canonical_food_compound_reference': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/outputs/reference/canonical_food_compound_reference.csv',
  'canonical_food_graph_features': '/Users/elchananmycheaux/Library/CloudStorage/OneDrive-Personal/Agens_Intelligens/Agent_Codex/Diet_Data_enhancement/output

## Mapping Confidence By Layer

In [3]:
def top_rows(df):
    return df.sort_values(['canonical_food_id', 'candidate_rank']).groupby('canonical_food_id', as_index=False).head(1)

rows = []
for name, df in [('Layer 1 nutrients', layer1), ('Layer 2 product/processing', layer2)]:
    top = top_rows(df)
    rows.append({
        'mapping_layer': name,
        'canonical_food_count': top['canonical_food_id'].nunique(),
        'candidate_rows': len(df),
        'high_confidence_percent': round(100 * (top['confidence'] == 'high').mean(), 2),
        'high_or_medium_percent': round(100 * top['confidence'].isin(['high', 'medium']).mean(), 2),
        'review_or_low_percent': round(100 * top['confidence'].isin(['review', 'low']).mean(), 2),
        'mean_best_score': round(pd.to_numeric(top['match_score'], errors='coerce').mean(), 4),
    })
validation_summary = pd.DataFrame(rows)
validation_summary.to_csv(VALIDATION / 'layered_mapping_validation_summary.csv', index=False)
validation_summary

,mapping_layer,canonical_food_count,candidate_rows,high_confidence_percent,high_or_medium_percent,review_or_low_percent,mean_best_score
0,Layer 1 nutrients,842,4068,20.19,33.37,66.63,0.6315
1,Layer 2 product/processing,842,4108,53.44,59.74,40.26,0.7762


## HITL Review Burden

In [4]:
hitl_summary = pd.DataFrame([
    {'metric': 'canonical_foods', 'value': len(hitl)},
    {'metric': 'pending_hitl_review', 'value': int((hitl['review_status'] == 'pending').sum())},
    {'metric': 'auto_accept_role_mappings', 'value': int((hitl['review_status'] == 'auto_accept').sum())},
    {'metric': 'potential_branded_or_packaged', 'value': int((hitl['suggested_identity_type'] == 'branded_or_packaged').sum())},
])
hitl_summary.to_csv(VALIDATION / 'layered_hitl_validation_summary.csv', index=False)
hitl_summary

,metric,value
0,canonical_foods,842
1,pending_hitl_review,798
2,auto_accept_role_mappings,44
3,potential_branded_or_packaged,503


## Feature Matrix Coverage

In [5]:
feature_metrics = pd.DataFrame([
    {'metric': 'canonical_food_rows', 'value': feature.shape[0]},
    {'metric': 'feature_columns', 'value': feature.shape[1]},
    {'metric': 'foods_with_openfoodfacts_match', 'value': int(feature.get('openfoodfacts_code', pd.Series(dtype=object)).notna().sum())},
    {'metric': 'foods_with_foodatlas_compounds', 'value': int((feature.get('foodatlas_compound_count', pd.Series([0]*len(feature))).fillna(0) > 0).sum())},
    {'metric': 'total_foodatlas_compound_links', 'value': int(feature.get('foodatlas_compound_count', pd.Series([0]*len(feature))).fillna(0).sum())},
])
feature_metrics.to_csv(VALIDATION / 'layered_feature_matrix_validation_summary.csv', index=False)
feature_metrics

,metric,value
0,canonical_food_rows,842
1,feature_columns,228
2,foods_with_openfoodfacts_match,840
3,foods_with_foodatlas_compounds,340
4,total_foodatlas_compound_links,47794


## Review Examples

In [6]:
cols = ['canonical_name', 'nutrient_matched_food_name', 'nutrient_confidence', 'product_matched_food_name', 'product_confidence', 'suggested_identity_type', 'review_reason']
hitl[cols].head(25)

,canonical_name,nutrient_matched_food_name,nutrient_confidence,product_matched_food_name,product_confidence,suggested_identity_type,review_reason
0,Beef tongue,"Beef, tongue, raw",high,beef,review,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
1,Beet Juice,Beet juice,high,Orange beet juice,low,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
2,Canned Tuna Fish,"Fish, tuna, canned",medium,Tuna Fish,low,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
3,Canned tomatoes,"Tomatoes, canned, cooked",medium,"Diced Tomatoes, Canned",low,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
4,Caviar,Caviar,high,Cowboy caviar,review,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
5,Celery Juice,Celery juice,high,Celery,review,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
6,Chard,"Chard, raw",medium,Organic Red Chard,review,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
7,Cheese Pastry,Cheese pastry puffs,medium,Cheese Danish Pastry,low,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
8,Chicken Liver,"Chicken, liver, raw",high,Smooth Chicken Liver Pate,low,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
9,Chicken drumstick,"Chicken, drumstick, lean, raw",medium,Chicken,review,generic_or_recipe,Auto-acceptable unless reviewer wants to inspect.
